# DS 4320 Project 1: Predicting the Men's March Madness Tournament

## Data Pipeline File

In [1]:
# Import packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import seaborn as sns
import re
import html
import time
import requests
from IPython.display import display
import duckdb
import logging

In [ ]:
# initiate logging
from fastapi import logger


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# Select completed seasons only; omit 2019-20 because no tournament was held
today = pd.Timestamp.today().normalize()
latest_completed_end_year = today.year if today.month >= 5 else today.year - 1
season_end_years = [y for y in range(2015, latest_completed_end_year + 1) if y != 2020]
# Runtime controls
MAX_SEASONS = None  # e.g., 3 for a quick test; None = all

# Reuse one HTTP session for reliability/performance
HTTP = requests.Session()
HTTP.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; DS4320/1.0)",
    "Accept": "application/json,text/html,*/*",
})


def season_label(end_year):
    return f"{end_year - 1}-{str(end_year)[-2:]}"


def flatten_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join([str(x) for x in c if str(x) != "nan"]).strip() for c in df.columns]
    return df


def find_col_by_terms(df, include_terms, exclude_terms=None):
    exclude_terms = exclude_terms or []
    for c in df.columns:
        lc = c.lower().strip()
        if all(term in lc for term in include_terms) and not any(term in lc for term in exclude_terms):
            logging.debug(f"Matched column '{c}' for include_terms={include_terms} exclude_terms={exclude_terms}")
            return c
    return None


TEAM_ALIASES = {
    "lsu": "louisiana state",
    "ole miss": "mississippi",
    "vcu": "virginia commonwealth",
    "n c central": "north carolina central",
    "northern ky": "northern kentucky",
}


def normalize_team_name(name):
    if pd.isna(name):
        return ""
    s = str(name).lower().strip()
    s = s.replace("&", "and")
    s = re.sub(r"\b(ncaa|nit|cbi|cit)\b", " ", s)
    s = re.sub(r"\buniv\.?\b", "university", s)
    s = re.sub(r"\bst\.?\b", "state", s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Remove institutional suffix words so 'Baylor' and 'Baylor University' match
    s = re.sub(r"\b(university|college)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return TEAM_ALIASES.get(s, s)


def round_to_stage(round_name):
    # NCAA round labels vary by season and may include HTML entities/registered symbols.
    r = html.unescape(str(round_name).lower())
    r = re.sub(r"[^a-z0-9 ]+", " ", r)
    r = re.sub(r"\s+", " ", r).strip()

    if not r:
        return ""
    if "first four" in r:
        return "First Four"
    if "round of 64" in r or "first round" in r:
        return "Round of 64"
    if "round of 32" in r or "second round" in r or "third round" in r:
        return "Round of 32"
    if "sweet 16" in r or "regional semifinal" in r:
        return "Sweet 16"
    if "elite eight" in r or "elite 8" in r or "regional final" in r:
        return "Elite 8"
    if "national semifinal" in r or "final four" in r:
        return "Final Four"
    if "championship" in r or "title" in r or "national final" in r:
        return "National Championship"
    return ""


def stage_rank(stage):
    order = {
        "First Four": 1,
        "Round of 64": 2,
        "Round of 32": 3,
        "Sweet 16": 4,
        "Elite 8": 5,
        "Final Four": 6,
        "National Championship": 7,
    }
    return order.get(stage, 0)


def fetch_net_table(end_year):
    url = f"https://www.warrennolan.com/basketball/{end_year}/net"
    net_df = pd.read_html(url)[0].copy()
    net_df.columns = [str(c).strip() for c in net_df.columns]
    if "Team" not in net_df.columns or "NET Rank" not in net_df.columns:
        raise ValueError(f"Unexpected NET table columns: {list(net_df.columns)}")
    out = pd.DataFrame({
        "Season": season_label(end_year),
        "Team": net_df["Team"].astype(str).str.strip(),
        "NETRank": pd.to_numeric(net_df["NET Rank"], errors="coerce")
    })
    out["TeamKey"] = out["Team"].map(normalize_team_name)
    return out


def fetch_ncaa_tournament_table(end_year):
    # Wider window captures play-in through title game across historical schedules.
    days = pd.date_range(f"{end_year}-03-10", f"{end_year}-04-20", freq="D")
    rows = []
    for d in days:
        url = f"https://data.ncaa.com/casablanca/scoreboard/basketball-men/d1/{d:%Y/%m/%d}/scoreboard.json"
        try:
            resp = HTTP.get(url, timeout=20)
            if resp.status_code != 200:
                continue
            data = resp.json()
            for entry in data.get("games", []):
                g = entry.get("game", {})
                stage = round_to_stage(g.get("bracketRound", ""))
                if stage == "":
                    continue
                for side in ["home", "away"]:
                    t = g.get(side, {}) or {}
                    names = t.get("names", {}) or {}
                    # Prefer short names for closer alignment with season/team tables
                    team_name = names.get("short") or names.get("full")
                    if not team_name:
                        continue
                    seed_raw = t.get("seed")
                    try:
                        seed = int(seed_raw) if str(seed_raw).strip() != "" else pd.NA
                    except Exception:
                        seed = pd.NA
                    rows.append({
                        "Season": season_label(end_year),
                        "Team": str(team_name).strip(),
                        "TeamKey": normalize_team_name(team_name),
                        "TournamentSeed": seed,
                        "RoundPlayed": stage,
                        "RoundRank": stage_rank(stage),
                    })
        except Exception:
            print(f"Error fetching/parsing tournament data for {end_year} on {d:%Y-%m-%d}")
            logging.error(f"Error occurred while fetching NCAA tournament data for {end_year}")
            continue

    if not rows:
        return pd.DataFrame(columns=["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"] )

    gdf = pd.DataFrame(rows)
    summary = gdf.groupby(["Season", "TeamKey"], as_index=False).agg(
        TournamentSeed=("TournamentSeed", "min"),
        FurthestRank=("RoundRank", "max"),
    )
    rank_to_name = {
        1: "First Four",
        2: "Round of 64",
        3: "Round of 32",
        4: "Sweet 16",
        5: "Elite 8",
        6: "Final Four",
        7: "National Championship",
    }
    summary["TournamentFinishRound"] = summary["FurthestRank"].map(rank_to_name)
    summary["MadeNCAATournament"] = True
    return summary[["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"]]


if MAX_SEASONS is not None:
    target_end_years = season_end_years[:MAX_SEASONS]
else:
    target_end_years = season_end_years

team_season_stats_rows = []
failed_seasons = []
net_rows = []
tourney_rows = []

overall_start = time.time()
for i, end_year in enumerate(target_end_years, start=1):
    label = season_label(end_year)
    season_start = time.time()
    print(f"[{i}/{len(target_end_years)}] Starting {label}...")
    logger.info(f"Starting data fetch for {label} (end_year={end_year})")

    try:
        url = f"https://www.basketball-reference.com/cbb/seasons/men/{end_year}-school-stats.html"
        season_df = pd.read_html(url)[0]
        season_df = flatten_columns(season_df)

        team_col = find_col_by_terms(season_df, ["school"] )
        wins_col = find_col_by_terms(season_df, ["overall", "w"], exclude_terms=["w-l%", "srs", "sos"] )
        losses_col = find_col_by_terms(season_df, ["overall", "l"], exclude_terms=["w-l%", "srs", "sos"] )
        winpct_col = find_col_by_terms(season_df, ["w-l%"] )
        games_col = find_col_by_terms(season_df, ["overall", "g"] )
        srs_col = find_col_by_terms(season_df, ["overall", "srs"] )
        sos_col = find_col_by_terms(season_df, ["overall", "sos"] )

        conf_w_col = find_col_by_terms(season_df, ["conf", "w"], exclude_terms=["w-l%"] )
        conf_l_col = find_col_by_terms(season_df, ["conf", "l"], exclude_terms=["w-l%"] )
        home_w_col = find_col_by_terms(season_df, ["home", "w"], exclude_terms=["w-l%"] )
        home_l_col = find_col_by_terms(season_df, ["home", "l"], exclude_terms=["w-l%"] )
        away_w_col = find_col_by_terms(season_df, ["away", "w"], exclude_terms=["w-l%"] )
        away_l_col = find_col_by_terms(season_df, ["away", "l"], exclude_terms=["w-l%"] )

        pts_col = find_col_by_terms(season_df, ["points", "tm"] )
        opp_pts_col = find_col_by_terms(season_df, ["points", "opp"] )

        mp_col = find_col_by_terms(season_df, ["totals", "mp"] )
        fg_col = find_col_by_terms(season_df, ["totals", "fg"], exclude_terms=["fg%", "fga"] )
        fga_col = find_col_by_terms(season_df, ["totals", "fga"] )
        fg_pct_col = find_col_by_terms(season_df, ["totals", "fg%"] )
        tp_col = find_col_by_terms(season_df, ["totals", "3p"], exclude_terms=["3p%", "3pa"] )
        tpa_col = find_col_by_terms(season_df, ["totals", "3pa"] )
        tp_pct_col = find_col_by_terms(season_df, ["totals", "3p%"] )
        ft_col = find_col_by_terms(season_df, ["totals", "ft"], exclude_terms=["ft%", "fta"] )
        fta_col = find_col_by_terms(season_df, ["totals", "fta"] )
        ft_pct_col = find_col_by_terms(season_df, ["totals", "ft%"] )
        orb_col = find_col_by_terms(season_df, ["totals", "orb"] )
        trb_col = find_col_by_terms(season_df, ["totals", "trb"] )
        ast_col = find_col_by_terms(season_df, ["totals", "ast"] )
        stl_col = find_col_by_terms(season_df, ["totals", "stl"] )
        blk_col = find_col_by_terms(season_df, ["totals", "blk"] )
        tov_col = find_col_by_terms(season_df, ["totals", "tov"] )
        pf_col = find_col_by_terms(season_df, ["totals", "pf"] )

        if team_col is None:
            raise ValueError(f"Could not find team column. Columns: {list(season_df.columns)}")

        season_df = season_df[season_df[team_col].astype(str).str.lower() != "school"]

        def num_or_na(col_name):
            if col_name is None:
                return pd.Series(pd.NA, index=season_df.index)
            return pd.to_numeric(season_df[col_name], errors="coerce")

        work = pd.DataFrame()
        work["Team"] = season_df[team_col].astype(str).str.strip()
        work["TeamKey"] = work["Team"].map(normalize_team_name)
        work["Wins"] = num_or_na(wins_col)
        work["Losses"] = num_or_na(losses_col)
        work["Games"] = num_or_na(games_col)
        work["SRS"] = num_or_na(srs_col)
        work["SOS"] = num_or_na(sos_col)
        work["ConfWins"] = num_or_na(conf_w_col)
        work["ConfLosses"] = num_or_na(conf_l_col)
        work["HomeWins"] = num_or_na(home_w_col)
        work["HomeLosses"] = num_or_na(home_l_col)
        work["AwayWins"] = num_or_na(away_w_col)
        work["AwayLosses"] = num_or_na(away_l_col)

        if games_col is None:
            work["Games"] = work["Wins"].fillna(0) + work["Losses"].fillna(0)

        work["PointsForTotal"] = num_or_na(pts_col)
        work["PointsAgainstTotal"] = num_or_na(opp_pts_col)
        denom = work["Games"].replace(0, pd.NA)
        work["AvgPointsFor"] = work["PointsForTotal"] / denom
        work["AvgPointsAgainst"] = work["PointsAgainstTotal"] / denom

        if winpct_col is not None:
            work["WinPct"] = pd.to_numeric(season_df[winpct_col], errors="coerce")
        else:
            w_denom = work["Wins"].fillna(0) + work["Losses"].fillna(0)
            work["WinPct"] = work["Wins"] / w_denom.replace(0, pd.NA)

        work["TotalsMP"] = num_or_na(mp_col)
        work["TotalsFG"] = num_or_na(fg_col)
        work["TotalsFGA"] = num_or_na(fga_col)
        work["TotalsFGPct"] = num_or_na(fg_pct_col)
        work["Totals3P"] = num_or_na(tp_col)
        work["Totals3PA"] = num_or_na(tpa_col)
        work["Totals3PPct"] = num_or_na(tp_pct_col)
        work["TotalsFT"] = num_or_na(ft_col)
        work["TotalsFTA"] = num_or_na(fta_col)
        work["TotalsFTPct"] = num_or_na(ft_pct_col)
        work["TotalsORB"] = num_or_na(orb_col)
        work["TotalsTRB"] = num_or_na(trb_col)
        work["TotalsAST"] = num_or_na(ast_col)
        work["TotalsSTL"] = num_or_na(stl_col)
        work["TotalsBLK"] = num_or_na(blk_col)
        work["TotalsTOV"] = num_or_na(tov_col)
        work["TotalsPF"] = num_or_na(pf_col)

        work["HomeGames"] = pd.NA
        work["AvgPTS_Box"] = pd.NA
        work["AvgREB_Box"] = pd.NA
        work["AvgAST_Box"] = pd.NA
        work["Season"] = label

        net_df = fetch_net_table(end_year)
        trn_df = fetch_ncaa_tournament_table(end_year)
        net_rows.append(net_df)
        tourney_rows.append(trn_df)
        print(f"    Tournament team keys for {label}: {len(trn_df)}")
        logger.info(f"Fetched data for {label}: {len(work)} teams, NET={len(net_df)}, Tournament={len(trn_df)}")

        team_season_stats_rows.append(work)
        elapsed = time.time() - season_start
        print(f"[{i}/{len(target_end_years)}] Finished {label} in {elapsed:.1f}s ({len(work)} teams)")
        logger.info(f"Completed data processing for {label} in {elapsed:.1f}s")
        time.sleep(0.15)

    except Exception as e:
        failed_seasons.append((label, str(e)))
        elapsed = time.time() - season_start
        print(f"[{i}/{len(target_end_years)}] {label} failed in {elapsed:.1f}s: {e}")
        logger.info(f"Data fetch failed for {label} after {elapsed:.1f}s: {e}")

if team_season_stats_rows:
    team_season_stats = pd.concat(team_season_stats_rows, ignore_index=True)
    logger.info(f"Combined team season stats: {len(team_season_stats)} rows across {team_season_stats['Season'].nunique()} seasons")
else:
    team_season_stats = pd.DataFrame()

if net_rows:
    net_rankings = pd.concat(net_rows, ignore_index=True).drop_duplicates(["Season", "TeamKey"], keep="first")
    logger.info(f"Combined NET rankings: {len(net_rankings)} rows across {net_rankings['Season'].nunique()} seasons")
else:
    net_rankings = pd.DataFrame(columns=["Season", "Team", "TeamKey", "NETRank"] )

if tourney_rows:
    ncaa_tournament_results = pd.concat(tourney_rows, ignore_index=True).drop_duplicates(["Season", "TeamKey"], keep="first")
    logger.info(f"Combined NCAA tournament results: {len(ncaa_tournament_results)} rows across {ncaa_tournament_results['Season'].nunique()} seasons")
else:
    ncaa_tournament_results = pd.DataFrame(columns=["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"] )

if not team_season_stats.empty:
    merged = team_season_stats.merge(
        net_rankings[["Season", "TeamKey", "NETRank"]],
        on=["Season", "TeamKey"],
        how="left"
    )
    merged = merged.merge(
        ncaa_tournament_results[["Season", "TeamKey", "MadeNCAATournament", "TournamentSeed", "TournamentFinishRound"]],
        on=["Season", "TeamKey"],
        how="left"
    )
    merged["MadeNCAATournament"] = merged["MadeNCAATournament"].fillna(False)
    merged = merged.sort_values(["Season", "Team"]).reset_index(drop=True)
    logger.info(f"Merged final dataset: {len(merged)} rows with {merged['Season'].nunique()} seasons and {merged['Team'].nunique()} teams")
else:
    merged = pd.DataFrame()

cbbpy_team_season_stats = merged

total_elapsed = time.time() - overall_start
print(f"\nRuntime this execution: {total_elapsed/60:.2f} minutes")
print(f"Seasons requested: {len(target_end_years)}")
print(f"Seasons loaded: {cbbpy_team_season_stats['Season'].nunique() if not cbbpy_team_season_stats.empty else 0}")
print(f"Team-season rows: {len(cbbpy_team_season_stats):,}")

if not cbbpy_team_season_stats.empty:
    display(cbbpy_team_season_stats.head(20))
    print("\nRelationship tables available:")
    print(f"- net_rankings rows: {len(net_rankings):,}")
    print(f"- ncaa_tournament_results rows: {len(ncaa_tournament_results):,}")
    print(f"- merged rows with MadeNCAATournament=True: {int(cbbpy_team_season_stats['MadeNCAATournament'].sum()):,}")
    print("\nTournament coverage by season:")
    print(
        cbbpy_team_season_stats.groupby("Season", as_index=False)["MadeNCAATournament"]
        .sum()
        .rename(columns={"MadeNCAATournament": "TournamentTeams"})
        .to_string(index=False)
    )

if failed_seasons:
    print("\nSeasons with issues:")
    for s, msg in failed_seasons:
        print(f"- {s}: {msg}")

[1/10] Starting 2014-15...
[1/10] 2014-15 failed in 1.5s: HTTP Error 404: Not Found
[2/10] Starting 2015-16...
[2/10] 2015-16 failed in 1.0s: HTTP Error 404: Not Found
[3/10] Starting 2016-17...
[3/10] 2016-17 failed in 0.9s: HTTP Error 404: Not Found
[4/10] Starting 2017-18...
[4/10] 2017-18 failed in 0.7s: HTTP Error 404: Not Found
[5/10] Starting 2018-19...


KeyboardInterrupt: 

#### Develop the Database using DuckDB

ERD: 

In [ ]:
# Ensure each team-season row has a stable primary key
if cbbpy_team_season_stats.empty:
    raise ValueError("cbbpy_team_season_stats is empty. Run the data creation cell first.")

cbbpy_team_season_stats = cbbpy_team_season_stats.copy()
cbbpy_team_season_stats["SquadID"] = (
    cbbpy_team_season_stats["Season"].astype(str) + "_" + cbbpy_team_season_stats["TeamKey"].astype(str)
)

# Load data into DuckDB tables
con = duckdb.connect(database="data/ncaam.db")
seasons = sorted(cbbpy_team_season_stats["Season"].dropna().unique().tolist())

# Core fact table
con.execute("""
    CREATE OR REPLACE TABLE team_season_stats AS
    SELECT * FROM cbbpy_team_season_stats
""")

# Dimension / relationship tables
con.execute("""
    CREATE OR REPLACE TABLE net_rankings AS
    SELECT DISTINCT Season, TeamKey, NETRank
    FROM net_rankings
""")
con.execute("""
    CREATE OR REPLACE TABLE ncaa_tournament_results AS
    SELECT DISTINCT Season, TeamKey, MadeNCAATournament, TournamentSeed, TournamentFinishRound
    FROM ncaa_tournament_results
""")

# Join table for season/team lookup
con.execute("""
    CREATE OR REPLACE TABLE join_table AS
    SELECT DISTINCT SquadID, TeamKey, Team, Season
    FROM cbbpy_team_season_stats
    ORDER BY Season, Team
""")

# Optional per-season tables for convenience
for season in seasons:
    season_df = cbbpy_team_season_stats[cbbpy_team_season_stats["Season"] == season]
    table_name = f"season_{season.replace('-', '_')}"
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM season_df
    """)
    logging.info(f"Loaded table {table_name} with {len(season_df)} rows")

logging.info("Data loading complete. Available tables:")
tables = con.execute("SHOW TABLES").fetchall()
for t in tables:
    logging.info(f"- {t[0]}")

#### Convert DB to Parquet files for easy storage

In [ ]:
# convert db tables to parquet files


for t in tables:
    table_name = t[0]
    df = con.execute(f"SELECT * FROM {table_name}").fetchdf()
    df.to_parquet(f"{table_name}.parquet")
    logging.info(f"Exported {table_name} to {table_name}.parquet")

